In [42]:
import pickle
import glob
import os
import subprocess
import pandas as pd
import numpy as np


In [43]:

odb_filenames = glob.glob('abq\\*.odb')
print('found odbs:', odb_filenames)


for odb_filename in odb_filenames:
    # abaqus python extract_from_odb.py .\abq\Job-2.odb --processes 32 --field_names NT11 S
    result = subprocess.run(["abaqus", "python", "extract_from_odb.py", odb_filename, "--processes", "16", "--field_names", "NT11", "S", "COORD"], shell=True, capture_output=True, text=True)
    
    print(result)

found odbs: ['abq\\Job-1.odb', 'abq\\Job-2.odb']
CompletedProcess(args=['abaqus', 'python', 'extract_from_odb.py', 'abq\\Job-1.odb', '--processes', '16', '--field_names', 'NT11', 'S', 'COORD'], returncode=0, stdout="Namespace(odb_filepath='abq\\\\Job-1.odb', step_numbers=None, field_names=['NT11', 'S', 'COORD'], processes=16)\nProcessing file: abq\\Job-1.odb\nStep: trans, Step number: 1, Number of frames: 11\nStep: final, Step number: 2, Number of frames: 5\n(step, frame) = [(1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (1, 10), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4)]\n[('abq\\\\Job-1.odb', [1], [0], ['NT11', 'S', 'COORD']), ('abq\\\\Job-1.odb', [1], [1], ['NT11', 'S', 'COORD']), ('abq\\\\Job-1.odb', [1], [2], ['NT11', 'S', 'COORD']), ('abq\\\\Job-1.odb', [1], [3], ['NT11', 'S', 'COORD']), ('abq\\\\Job-1.odb', [1], [4], ['NT11', 'S', 'COORD']), ('abq\\\\Job-1.odb', [1], [5], ['NT11', 'S', 'COORD']), ('abq\\\\Job-1.odb', [1], [6], ['NT11', 'S', 'COORD']

In [49]:
data_files = glob.glob('abq\\Job-1\\*.pkl')



def process_pkl_files(data_files):

    data =[]
    for data_file in data_files:
        df = pd.read_pickle(data_file)
        data.append(df)
    print(f'processing {len(data)} files')


    inds = [df.index[0]  for df in data]
    field_names = [df.columns.get_level_values(0)[0] for df in data]

    df_meta = pd.DataFrame(inds, columns = df.index.names)
    df_meta['field_name'] = field_names
    G = df_meta.groupby(by=df_meta.columns.tolist())

    data_dict = {}
    for name, group in G:

        indicies = group.index.to_list()

        df_tmp = pd.concat([data[i] for i in indicies], axis = 1)
        field_name = df_tmp.columns.get_level_values(0)[0]

        if field_name not in data_dict:
            data_dict[field_name] = []
        data_dict[field_name].append(df_tmp)

    for field_name, df_list in data_dict.items():

        df_tmp = pd.concat(df_list, axis = 0)
        df_tmp.sort_index(inplace = True, axis = 0)
        df_tmp.sort_index(inplace = True, axis = 1)
        df_tmp = df_tmp.astype(np.float32)
        assert df_tmp.notna().all().all(), "Some values are missing"


        data_dict[field_name] = df_tmp

    for field_name, df in data_dict.items():
        print(f'{field_name}: {df.shape}')

    return data_dict


data_dict = process_pkl_files(data_files)

processing 128 files
COORD: (16, 1968)
NT11: (16, 400)
S: (16, 1536)


In [ ]:
for field_name, df in data_dict.items():
    print(df.head())
    
    #df.to_csv(f'{field_name}.csv')

field_name                                            COORD            \
component                                             COOR1             
instance_name                                      PART-1-1             
node_or_element_idx                                     1         2     
integration_point                                        -1        -1   
step_number increment_number total_time step_name                       
1           0                0.0        trans      9.009688  6.234898   
            1                1.0        trans      9.009688  6.234898   
            2                2.0        trans      9.009688  6.234898   
            3                3.0        trans      9.009688  6.234898   
            4                4.0        trans      9.009688  6.234898   

field_name                                                             \
component                                                               
instance_name                                     